In [1]:
#基础导入
import shutil
import gc
import os
import re
import json
import csv
import html
from pathlib import Path
from typing import TypedDict, List, Dict, Optional, Tuple
from collections import defaultdict
from difflib import SequenceMatcher

from langgraph.graph import StateGraph, END
from langchain_openai import ChatOpenAI
from langchain_core.language_models import LLM
from langchain_core.documents import Document
from langchain_chroma import Chroma
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_core.messages import SystemMessage, HumanMessage


d:\anaconda\envs\mscpy\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
#清理 Chroma 缓存（避免向量库冲突）
print("=== 清理旧向量库 ===")
db_paths = [
    "./chroma_db",
    "./chroma_db/product_vectors",
    "./chroma_db/rule_vectors",
    "./chroma_db/nursing_vectors",
]
for path in db_paths:
    if Path(path).exists():
        try:
            shutil.rmtree(path)
            print(f"已清理: {path}")
        except Exception as e:
            print(f"清理 {path} 失败: {e}")

globals_to_clear = [
    'product_vectorstore', 'rule_vectorstore',
    'nursing_vectorstore', 'embeddings'
]
for var in globals_to_clear:
    if var in globals():
        del globals()[var]
        print(f"已释放全局变量: {var}")
gc.collect()
print("=== 向量库清理完成 ===\n")


=== 清理旧向量库 ===
已清理: ./chroma_db
=== 向量库清理完成 ===



In [ ]:
#Embedding 初始化
print("=== 初始化 Embedding ===")
os.environ["HF_ENDPOINT"] = "https://hf-mirror.com"
os.environ["HUGGINGFACE_HUB_SYMLINKS"] = "0"

model_name = "BAAI/bge-m3"
cache_dir = "./model_cache"
embeddings = HuggingFaceEmbeddings(
    model_name=model_name,
    cache_folder=cache_dir
)
print(f"Embedding 模型加载完成: {model_name}\n")


=== 初始化 Embedding ===


In [ ]:
#Logic Rule 加载 & 向量库构建
print("=== 加载 Logic Rule ===")

def load_logic_rules(json_path: str) -> List[Document]:
    if not Path(json_path).exists():
        raise FileNotFoundError(f"找不到逻辑规则文件: {json_path}")
    with open(json_path, 'r', encoding='utf-8') as f:
        logic_data = json.load(f)
    nodes = logic_data.get('nodes', {})
    documents = []
    for node_id, node_content in nodes.items():
        if not isinstance(node_content, dict):
            continue
        node_type = node_content.get('type', '')
        if node_type in ['single_choice', 'multi_choice']:
            question = node_content.get('question', '')
            options = node_content.get('options', [])
            next_steps = node_content.get('next', {})
            options_text = " | ".join([
                f"选项'{opt}'→{next_steps.get(opt, '无跳转')}"
                for opt in options if opt in next_steps
            ])
            page_content = f"""问题:{question}
节点ID:{node_id} 选项:{','.join(options)} 跳转逻辑:{options_text}"""
            metadata = {
                "source": "02_LOGIC_RULE",
                "node_id": node_id,
                "node_type": node_type,
                "options_count": len(options),
                "has_recommendation": any(
                    v.startswith('recommend_') for v in next_steps.values()
                ),
                "next_steps": json.dumps(next_steps, ensure_ascii=False),
                "page_content": page_content
            }
            documents.append(Document(page_content=question, metadata=metadata))

    recommend_nodes = logic_data.get('recommend_nodes', {})
    for node_id, node_content in recommend_nodes.items():
        if not isinstance(node_content, dict):
            continue
        recommendation = node_content.get('recommendation', node_id.replace('recommend_', ''))
        products = node_content.get('products', [])
        page_content = f"""
推荐节点ID:{node_id}
推荐内容:{recommendation}
关联产品:{','.join(products) if products else '无'}
"""
        metadata = {
            "source": "02_LOGIC_RULE",
            "node_id": node_id,
            "node_type": "recommendation",
            "recommendation_category": recommendation,
            "page_content": page_content
        }
        documents.append(Document(page_content=recommendation, metadata=metadata))
    print(f"Logic Rule 加载完成: {len(documents)} 条文档")
    return documents

rule_json_path = "02_LOGIC_RULE.json"
rule_docs = []
try:
    rule_docs = load_logic_rules(rule_json_path)
except FileNotFoundError as e:
    print(f"警告: {e}，将使用空规则库继续")

if rule_docs:
    os.makedirs("./chroma_db/rule_vectors", exist_ok=True)
    rule_vectorstore = Chroma.from_documents(
        documents=rule_docs,
        embedding=embeddings,
        persist_directory="./chroma_db/rule_vectors",
        collection_name="erent_logic_rules"
    )
    print(f"规则向量库构建完成: {len(rule_docs)} 条\n")
else:
    rule_vectorstore = None
    print("规则向量库为空\n")


=== 加载 Logic Rule ===
Logic Rule 加载完成: 49 条文档
规则向量库构建完成: 49 条



In [ ]:
#LLM 初始化 (OpenRouter)
print("=== 初始化 LLM ===")
api_key = os.getenv("OPENROUTER_API_KEY", ) #自行添加api key
if not api_key or "sk-or-v1-" not in api_key:
    print("警告: 未检测到有效的 OpenRouter API Key，将使用 MockLLM")
    api_key = ""

candidate_models = [
    "openai/gpt-4o",
    "anthropic/claude-sonnet-4-6",
    "openai/gpt-4o-mini",
    "meta-llama/llama-3.1-70b-instruct",
    "google/gemini-2.0-flash-001"
]

llm = None
last_error = None
if api_key:
    for model_name in candidate_models:
        try:
            print(f"尝试连接: {model_name}...")
            test_llm = ChatOpenAI(
                model=model_name,
                openai_api_base="https://openrouter.ai/api/v1",
                openai_api_key=api_key,
                temperature=0.2,
                max_tokens=2000,
                timeout=10,
            )
            test_llm.invoke("你好")
            llm = test_llm
            print(f"LLM 连接成功: {model_name}\n")
            break
        except Exception as e:
            print(f"连接失败: {str(e)[:100]}")
            last_error = e
            continue

if llm is None:
    print("\n所有模型连接失败，回退到 MockLLM")
    class MockLLM(LLM):
        def _call(self, prompt, stop=None, **kwargs):
            user_input = prompt.split("用户输入:")[-1].split("\n")[0].strip()
            if "护理" in user_input or "咨询" in user_input or "照顾" in user_input:
                return "护理咨询"
            elif "设备" in user_input or "租赁" in user_input or "器材" in user_input:
                return "设备租赁"
            elif "产品" in user_input or "浏览" in user_input or "看看" in user_input:
                return "产品浏览"
            else:
                return "护理咨询"
        @property
        def _llm_type(self):
            return "mock"
    llm = MockLLM()


=== 初始化 LLM ===
尝试连接: openai/gpt-4o...
LLM 连接成功: openai/gpt-4o



In [ ]:
#护理咨询 Prompts (来自 1nursing.pdf)
NURSING_CONSULT_SYSTEM_PROMPT = """# 角色：专业长者护理顾问
# 核心任务：你是一名专业的护理顾问，仅基于提供的【参考知识】解答用户的长者护理、辅具使用规范及健康照护问题。

# 强制约束（必须100%遵守）：
1. 你的回答必须严格基于提供的【参考知识】，绝对不允许编造或引入外部未经证实的知识。如果【参考资料】中没有相关信息，请诚实回答“手册中未提及相关细节”，不要胡编乱造。
2. 请尽量保留手册中的专业术语和具体操作步骤（如穿衣顺序、沟通技巧）。
3. 如果【参考知识】中没有相关信息，请直接回答：“抱歉，目前的护理知识库中暂未收录关于此问题的资料。建议您咨询专业医生或复康护士。”
4. 语言需温和、专业、富有同理心，尽量使用通俗易懂的口语化表达，方便长者及其照顾者理解。
5. 回答需条理清晰（可适当使用序号或分段），针对用户的痛点给出实用的建议。
6. 绝对不进行商业产品推销，仅提供护理和辅具的“使用指导”或“保养建议”。
7.【安全护栏 1 - 禁止医疗诊断】绝对不提供任何医疗诊断、疾病判断、病因分析、治疗方案、用药建议、偏方推荐，不评价用户或长者的病情严重程度，不预测病情发展。若用户询问疾病诊断、治疗、用药、病情判断等相关问题，必须直接回复：“抱歉，我无法提供医疗诊断、治疗及用药相关建议，请您咨询专业医生或复康护士，获取专业医疗指导。
8.【安全护栏 2 - 禁止违规内容】不回应任何敏感、负面、违规提问，包括但不限于：恶意诋毁辅具 / 平台、询问危险操作（自制危险辅具）、诱导负面情绪（如 “拖累家人” 相关吐槽）、无关闲聊（八卦、旅游、理财等）、占卜运势等，此类问题直接回复：“抱歉，我无法回应此类内容，请您咨询与长者护理、辅具使用相关的问题。
9.【安全护栏 3 - 合规声明】所有回答末尾必须添加固定合规声明，不可省略：“ 温馨提示：以上建议仅供长者护理参考，不替代专业医疗意见，如有健康相关疑问，请及时咨询医护人员。
10.【安全护栏 4 - 避免幻觉】不编造不存在的护理方法、辅具使用技巧、健康知识，不夸大护理效果，严格贴合【参考知识】内容，若不确定是否有相关信息，按第 3 条规则回复。"""

NURSING_CONSULT_USER_TEMPLATE = """【参考知识】
{context}

【用户问题】
{user_input}

请根据以上参考知识，解答用户的疑问："""


In [ ]:
#AgentState 定义（合并版）& 意图分类 Prompt
class AgentState(TypedDict):
    messages: list
    user_input: str
    intent: str
    is_intent_clear: bool
    logic_rule_qa_history: list
    collected_recommendations: list
    followup_results: list
    browse_selected_category: str
    browse_viewed_products: list
    nursing_qa_history: list

INTENT_CLASSIFY_PROMPT = """角色：长者辅具Chatbot 多级意图识别专家							
# 核心任务：仅判断用户输入的意图，严格按规则输出分类结果，不做任何额外回答、不解释、不闲聊	
# 前置安全护栏规则（优先执行，优先级高于意图分类）：
若用户输入属于以下任一情况，不做意图分类，直接固定输出：无法提供相关回答，请重新输入
1、询问疾病诊断、病因判断、病情严重程度、治疗方案、用药建议、偏方调理等医疗相关问题
2、涉及敏感低俗、负面消极、自残轻生、恶意吐槽诋毁等违规内容
3、无关闲聊、娱乐八卦、旅游理财、占卜运势、开发身份询问等与长者护理 / 辅具无关的问题
4、诱导推荐三无危险辅具、自制危险护理设备等不合规需求						
# 输出规则（必须严格四选一，仅输出分类结果，禁止输出其他内容）：							
1. 护理咨询：用户询问护理方法、辅具使用规范、照顾技巧、长者健康相关问题（如「怎么帮老人翻身」「轮椅怎么保养」）							
2. 产品-问题解决型：用户有具体痛点/需求，需要个性化推荐适配辅具（如「我妈妈走路不稳」「老人起不来床」「My mom can't sleep」「推荐什么辅具」）							
3. 产品-浏览了解型：用户仅想了解产品、查看品类/型号，无具体痛点（如「你们有什么床」「What beds do you have?」「我想看看轮椅」）							
4. 意图不清：无法明确判断用户意图，或用户输入模糊、无有效信息							

强制约束（必须100%遵守）							
1. 先执行前置安全护栏规则，命中违规直接输出固定话术；合规输入再按 4 类意图严格分类，绝对不自由发挥、不自创分类					
2. 支持粤语、繁体中文、口语化表达、语音转文字内容、长者常用表述							
3. 若用户同时提到产品和护理，优先按主意图分类（如「我想看看护理床，怎么帮老人用」→ 主意图为产品-浏览了解型，护理问题后续单独处理）							
4. 仅输出分类结果，禁止输出思考过程、解释、额外内容							
5. 绝对不回答用户问题，仅做意图分类							

用户输入：{user_input}							

请严格按规则输出分类结果："""


In [ ]:
# LogicRuleEngine 类
class LogicRuleEngine:
    def __init__(self, json_path: str):
        with open(json_path, 'r', encoding='utf-8') as f:
            self.rule_data = json.load(f)
        self.nodes = self.rule_data.get('nodes', {})
        self.recommend_nodes = self.rule_data.get('recommend_nodes', {})
        self.meta = self.rule_data.get('meta', {})
        self.start_node = self.meta.get('start_node', 'node_01_role')
        print(f"LogicRuleEngine 初始化完成")
        print(f"起始节点: {self.start_node}")
        print(f"问题节点: {len(self.nodes)} | 推荐节点: {len(self.recommend_nodes)}")

    def is_recommend_node(self, node_id: str) -> bool:
        return node_id in self.recommend_nodes

    def get_question_node(self, node_id: str) -> Optional[dict]:
        return self.nodes.get(node_id)

    def get_next_nodes(self, node_id: str, selected_options: List[str]) -> List[str]:
        node = self.nodes.get(node_id)
        if not node:
            return []
        next_map = node.get('next', {})
        next_nodes = []
        for option in selected_options:
            if '*' in next_map:
                nid = next_map['*']
            else:
                nid = next_map.get(option, '')
            if nid and nid not in next_nodes:
                next_nodes.append(nid)
        return next_nodes

    def get_recommendation_summary(self, recommend_ids: List[str]) -> List[dict]:
        result = []
        seen = set()
        for rid in recommend_ids:
            node = self.recommend_nodes.get(rid)
            if node and rid not in seen:
                seen.add(rid)
                result.append({
                    'node_id': rid,
                    'content': node.get('content', rid),
                    'device_tag': node.get('device_tag', ''),
                    'category_id': node.get('category_id', '')
                })
        return result

# 初始化引擎（如果文件存在）
logic_engine = None
if Path(rule_json_path).exists():
    logic_engine = LogicRuleEngine(rule_json_path)


LogicRuleEngine 初始化完成
起始节点: node_01_role
问题节点: 31 | 推荐节点: 18


In [ ]:
#节点逻辑运行 & LLM 选项匹配辅助函数
def run_node_logic(node_doc: Document, user_selected_options: list) -> List[str]:
    node_type = node_doc.metadata["node_type"]
    next_node_ids = []
    if node_type == "single_choice":
        selected = user_selected_options[0]
        next_id = get_next_node_id(node_doc, selected)
        next_node_ids.append(next_id)
    elif node_type == "multi_choice":
        for selected in user_selected_options:
            next_id = get_next_node_id(node_doc, selected)
            next_node_ids.append(next_id)
    print(f"节点 {node_type} 跳转: {next_node_ids}")
    return next_node_ids

def get_next_node_id(node_doc: Document, option: str) -> str:
    next_steps_str = node_doc.metadata.get("next_steps", "{}")
    next_steps = json.loads(next_steps_str)
    return next_steps.get(option, "")

OPTION_MATCH_SYSTEM_PROMPT = """你是一个选项匹配助手。请根据用户输入，从给定选项中选出最匹配的一项或多项。
要求：
1. 只输出匹配到的选项文本，多个用 ||| 分隔
2. 如果都不匹配，输出"无匹配"
3. 不要输出解释、序号或额外文字"""

OPTION_MATCH_USER_TEMPLATE = """当前问题: {current_question}
选项类型: {node_type_hint}
可选选项:
{options_numbered}
用户输入: {user_input}
请输出匹配选项:"""

def llm_match_options(user_input: str, current_question: str,
                      options: List[str], node_type: str) -> Tuple[List[str], bool]:
    node_type_hint = "单选" if node_type == "single_choice" else "多选"
    options_numbered = "\n".join([f"{i+1}. {opt}" for i, opt in enumerate(options)])
    user_prompt = OPTION_MATCH_USER_TEMPLATE.format(
        current_question=current_question,
        node_type_hint=node_type_hint,
        options_numbered=options_numbered,
        user_input=user_input
    )
    try:
        messages = [
            SystemMessage(content=OPTION_MATCH_SYSTEM_PROMPT),
            HumanMessage(content=user_prompt)
        ]
        raw = llm.invoke(messages).content.strip()
        if raw == "无匹配" or not raw:
            return [], False
        parts = [p.strip() for p in raw.split("|||")]
        matched = []
        for part in parts:
            if part in options:
                matched.append(part)
            else:
                for opt in options:
                    if opt in part or part in opt:
                        if opt not in matched:
                            matched.append(opt)
                        break
        if node_type == "single_choice" and len(matched) > 1:
            matched = matched[:1]
        return matched, bool(matched)
    except Exception as e:
        print(f"LLM 匹配失败: {e}")
        return [], False

def is_numeric_input(text: str) -> bool:
    cleaned = re.sub(r'[,，\s]+', '', text.strip())
    return bool(cleaned) and all(t.isdigit() for t in cleaned.split())

def parse_number_input(text: str, options: List[str], node_type: str) -> List[str]:
    numbers = re.findall(r'\d+', text)
    selected = []
    for n in numbers:
        idx = int(n) - 1
        if 0 <= idx < len(options) and options[idx] not in selected:
            selected.append(options[idx])
    if node_type == "single_choice" and len(selected) > 1:
        selected = selected[:1]
    return selected

def resolve_user_input(user_input: str, question: str,
                       options: List[str], node_type: str) -> Tuple[List[str], str]:
    if is_numeric_input(user_input):
        selected = parse_number_input(user_input, options, node_type)
        if selected:
            return selected, "number"
    matched, ok = llm_match_options(user_input, question, options, node_type)
    if ok:
        return matched, "llm"
    return [], "unclear"


In [ ]:
#Logic Rule 会话运行
class LogicRuleState(TypedDict):
    current_node: str
    pending_branches: list
    collected_recommendations: list
    qa_history: list
    is_complete: bool

def init_logic_rule_state() -> LogicRuleState:
    return {
        "current_node": logic_engine.start_node if logic_engine else "",
        "pending_branches": [],
        "collected_recommendations": [],
        "qa_history": [],
        "is_complete": False
    }

ASSESSMENT_BOT_SYSTEM_PROMPT = """你是一位专业的设备租赁评估助手。请根据 Logic Rule 的节点信息，以亲切、专业的方式向用户呈现问题和推荐。
要求：
1. 使用繁体中文
2. 清晰呈现问题和选项
3. 对于推荐节点，给出温暖的推荐语，并提及设备标签
4. 保持简洁，不要暴露内部节点 ID"""

ASSESSMENT_BOT_USER_TEMPLATE = """当前节点: {current_node}
问题内容: {question}
选项类型提示: {node_type_hint}
选项列表:
{options_text}
用户历史回答:
{user_answer_history}

请生成面向用户的自然语言呈现。"""

def llm_present_question(node_id: str, question: str, options: List[str],
                         node_type: str, qa_history: list) -> str:
    node_type_hint = (
        "可多选，请回复选项编号如 1,2,3"
        if node_type == "multi_choice"
        else "请回复一个选项编号如 1"
    )
    options_text = "\n".join([f"{i+1}. {opt}" for i, opt in enumerate(options)])
    if qa_history:
        history_lines = []
        for qa in qa_history:
            q_short = qa["question"][:20] + "..."
            a_text = "、".join(qa["selected"])
            history_lines.append(f"问题: {q_short} → 回答: {a_text}")
        user_answer_history = "\n".join(history_lines)
    else:
        user_answer_history = "（无）"
    user_prompt = ASSESSMENT_BOT_USER_TEMPLATE.format(
        current_node=node_id,
        question=question,
        node_type_hint=node_type_hint,
        options_text=options_text,
        user_answer_history=user_answer_history
    )
    try:
        messages = [
            SystemMessage(content=ASSESSMENT_BOT_SYSTEM_PROMPT),
            HumanMessage(content=user_prompt)
        ]
        result = llm.invoke(messages).content.strip()
        return result
    except Exception as e:
        print(f"LLM 生成问题展示失败: {e}，使用 fallback")
        fallback = (
            f"\n{'='*50}\n"
            f"{question}\n"
            f"{node_type_hint}\n\n"
            f"{options_text}\n"
            f"{'='*50}"
        )
        return fallback

def llm_present_recommendation(recommend_content: str, device_tag: str) -> str:
    user_prompt = f"""推荐内容: {recommend_content}
设备标签: {device_tag}

请生成面向用户的推荐语。"""
    try:
        messages = [
            SystemMessage(content=ASSESSMENT_BOT_SYSTEM_PROMPT),
            HumanMessage(content=user_prompt)
        ]
        result = llm.invoke(messages).content.strip()
        return result
    except Exception as e:
        return f"根据您的需求，我们推荐: {recommend_content}（设备标签: {device_tag}）"

def run_logic_rule_session() -> LogicRuleState:
    lr = init_logic_rule_state()
    print("\n" + "="*55)
    print("开始设备租赁评估会话")
    print("="*55)
    while not lr["is_complete"]:
        current = lr["current_node"]
        if not current:
            break
        # 1. 检查是否为推荐节点
        if logic_engine and logic_engine.is_recommend_node(current):
            if current not in lr["collected_recommendations"]:
                lr["collected_recommendations"].append(current)
            rec = logic_engine.recommend_nodes[current]
            output = llm_present_recommendation(
                recommend_content=rec.get('content', current),
                device_tag=rec.get('device_tag', '')
            )
            print(f"\n{output}")
            if lr["pending_branches"]:
                lr["current_node"] = lr["pending_branches"].pop(0)
                print(f"\n继续处理分支... 剩余 {len(lr['pending_branches'])} 个")
                continue
            else:
                lr["is_complete"] = True
                break
        # 2. 问题节点
        node = logic_engine.get_question_node(current) if logic_engine else None
        if not node:
            if lr["pending_branches"]:
                lr["current_node"] = lr["pending_branches"].pop(0)
            else:
                lr["is_complete"] = True
            break
        question = node.get("question", "")
        options = node.get("options", [])
        node_type = node.get("type", "single_choice")
        question_display = llm_present_question(
            node_id=current, question=question, options=options,
            node_type=node_type, qa_history=lr["qa_history"]
        )
        print(f"\n{question_display}")
        selected = []
        while not selected:
            raw = input("\n请输入您的选择: ").strip()
            if not raw:
                print("输入不能为空，请重新输入")
                continue
            selected, method = resolve_user_input(raw, question, options, node_type)
            if method == "unclear":
                print("\n未能理解您的输入，请重新选择")
                repeat = llm_present_question(
                    node_id=current, question=question, options=options,
                    node_type=node_type, qa_history=lr["qa_history"]
                )
                print(f"\n{repeat}")
                selected = []
                continue
            if method == "llm":
                matched_text = "、".join(selected)
                print(f"\n系统理解您的选择为: {matched_text}")
                confirm = input("确认请按 Enter，重新选择请输入 N: ").strip().lower()
                if confirm in ("n", "no", "否", "不"):
                    selected = []
                    repeat = llm_present_question(
                        node_id=current, question=question, options=options,
                        node_type=node_type, qa_history=lr["qa_history"]
                    )
                    print(f"\n{repeat}")
                    continue
        print(f"确认选择: {' | '.join(selected)}")
        next_nodes = logic_engine.get_next_nodes(current, selected) if logic_engine else []
        lr["qa_history"].append({
            "node_id": current,
            "question": question,
            "options": options,
            "node_type": node_type,
            "selected": selected,
            "next_nodes": next_nodes
        })
        if not next_nodes:
            if lr["pending_branches"]:
                lr["current_node"] = lr["pending_branches"].pop(0)
            else:
                lr["is_complete"] = True
        elif len(next_nodes) == 1:
            lr["current_node"] = next_nodes[0]
        else:
            lr["current_node"] = next_nodes[0]
            for n in next_nodes[1:]:
                if n not in lr["pending_branches"] and n != lr["current_node"]:
                    lr["pending_branches"].append(n)
    _display_final_recommendations(lr)
    return lr

def _display_final_recommendations(lr: LogicRuleState):
    print("\n" + "="*55)
    print("评估完成，问答摘要")
    for i, qa in enumerate(lr["qa_history"], 1):
        q_short = qa["question"][:28] + ("..." if len(qa["question"]) > 28 else "")
        a_text = " | ".join(qa["selected"])
        print(f"Q{i}: {q_short}")
        print(f"A{i}: {a_text}")
    disclaimer = logic_engine.meta.get("global_disclaimer", "") if logic_engine else ""
    if disclaimer:
        print("\n" + disclaimer.strip())
    print("="*55)


In [ ]:
#ProductBrowseEngine 类 (来自 1nursing.pdf)
class ProductBrowseEngine:
    def __init__(self, product_json_path: str, category_csv_path: str):
        self.product_json_path = product_json_path
        self.category_csv_path = category_csv_path
        self.product_details = self._load_product_details(product_json_path)
        self.category_rows = self._load_category_rows(category_csv_path)
        self.catalog = self._build_catalog()
        self.category_names = sorted(self.catalog.keys())
        print("ProductBrowseEngine 初始化完成")
        print(f"产品分类数: {len(self.category_names)}")
        print(f"总产品数: {sum(len(items) for items in self.catalog.values())}")

    def _normalize_text(self, text: str) -> str:
        text = (text or "").strip().lower()
        replacements = {
            "（": "(", "）": ")", "-": "", "_": "", "，": "", ",": "", " ": "",
        }
        for old, new in replacements.items():
            text = text.replace(old, new)
        return "".join(text.split())

    def _strip_html(self, text: str) -> str:
        if not text:
            return ""
        cleaned = re.sub(r"<br\s*/?>", "\n", text, flags=re.IGNORECASE)
        cleaned = re.sub(r"</p>|</li>|</h\d>", "\n", cleaned, flags=re.IGNORECASE)
        cleaned = re.sub(r"<[^>]+>", "", cleaned)
        cleaned = html.unescape(cleaned)
        cleaned = re.sub(r"\s+", "", cleaned)
        return cleaned.strip()

    def _load_product_details(self, json_path: str) -> list:
        with open(json_path, "r", encoding="utf-8") as f:
            return json.load(f)

    def _load_category_rows(self, csv_path: str) -> list:
        with open(csv_path, "r", encoding="utf-8-sig", newline="") as f:
            return list(csv.DictReader(f))

    def _best_detail_match(self, product_name: str) -> dict:
        normalized_target = self._normalize_text(product_name)
        best_match = None
        best_score = 0.0
        for item in self.product_details:
            candidate_name = item.get("Name", "")
            normalized_candidate = self._normalize_text(candidate_name)
            if not normalized_candidate:
                continue
            if normalized_target == normalized_candidate:
                return item
            if normalized_target in normalized_candidate or normalized_candidate in normalized_target:
                score = 0.95
            else:
                score = SequenceMatcher(None, normalized_target, normalized_candidate).ratio()
            if score > best_score:
                best_score = score
                best_match = item
        return best_match if best_score >= 0.62 else {}

    def _build_catalog(self) -> dict:
        catalog = defaultdict(list)
        for row in self.category_rows:
            product_name = row.get("product_name", "").strip()
            category_name = row.get("category_name", "未分类").strip() or "未分类"
            detail = self._best_detail_match(product_name)
            description = (
                self._strip_html(detail.get("eCommerce Description", ""))
                or (detail.get("Product/Description") or "").strip()
                or (row.get("description") or "").strip()
            )
            product_record = {
                "name": product_name,
                "category_name": category_name,
                "category_id": row.get("category_id", ""),
                "stock_status": row.get("stock_status", ""),
                "in_stock": str(row.get("in_stock", "")).lower() == "true",
                "sales_price": detail.get("Sales Price", ""),
                "quantity_on_hand": detail.get("Quantity On Hand", ""),
                "description": description,
                "video_url": detail.get("Introduction Video URL", ""),
                "dimension": {
                    "height": detail.get("Dimension Height") or row.get("dimension_height") or 0,
                    "length": detail.get("Dimension Length") or row.get("dimension_length") or 0,
                    "width": detail.get("Dimension Width") or row.get("dimension_width") or 0,
                },
                "net_weight": detail.get("Net Weight") or row.get("net_weight") or 0,
                "raw_detail": detail,
            }
            catalog[category_name].append(product_record)
        for category_name, items in catalog.items():
            items.sort(key=lambda item: (not item["in_stock"], item["name"]))
        return dict(catalog)

    def get_categories(self) -> List[str]:
        return list(self.category_names)

    def match_category(self, user_input: str) -> Tuple[Optional[str], str]:
        categories = self.get_categories()
        selected, method = resolve_user_input(user_input, "请选择分类", categories, "single_choice")
        if selected:
            return selected[0], method
        normalized_input = self._normalize_text(user_input)
        for category in categories:
            normalized_category = self._normalize_text(category)
            if normalized_input and (normalized_input in normalized_category or normalized_category in normalized_input):
                return category, "fallback"
        return None, "unclear"

    def list_products(self, category_name: str) -> List[dict]:
        return self.catalog.get(category_name, [])

    def format_product_list(self, category_name: str, max_items: int = 12) -> str:
        products = self.list_products(category_name)
        if not products:
            return f"\n【{category_name}】暂无产品"
        lines = [f"\n【{category_name}】产品列表："]
        for idx, product in enumerate(products[:max_items], 1):
            price_text = f"HK${product['sales_price']:.0f}" if isinstance(product.get("sales_price"), (int, float)) else "价格待定"
            stock_text = "有货" if product.get("in_stock") else "缺货/预订"
            lines.append(f"{idx}. {product['name']} | {price_text} | {stock_text}")
        if len(products) > max_items:
            lines.append(f"...... 还有 {len(products)-max_items} 款产品，输入 0 返回分类，q 退出")
        lines.append("输入产品编号查看详情，0 返回分类，q 退出")
        return "\n".join(lines)

    def _brief_description(self, description: str, limit: int = 140) -> str:
        if not description:
            return "暂无描述"
        return description if len(description) <= limit else description[:limit].rstrip() + "..."

    def format_product_detail(self, product: dict) -> str:
        dimension = product.get("dimension", {})
        size_parts = []
        if dimension.get("length"):
            size_parts.append(f"长{dimension['length']}")
        if dimension.get("width"):
            size_parts.append(f"宽{dimension['width']}")
        if dimension.get("height"):
            size_parts.append(f"高{dimension['height']}")
        size_text = "/".join(size_parts) if size_parts else "暂无尺寸"
        price = product.get("sales_price")
        price_text = f"HK${price:.0f}" if isinstance(price, (int, float)) else "价格待定"
        stock_text = "有货" if product.get("in_stock") else "缺货/预订"
        weight = product.get("net_weight")
        weight_text = f"{weight}kg" if isinstance(weight, (int, float)) and weight not in (0, 0.0, "") else "暂无重量"
        description = self._brief_description(product.get("description", ""), limit=360)
        lines = [
            "\n" + "="*60,
            f"产品名称: {product['name']}",
            f"所属分类: {product['category_name']}",
            f"销售价格: {price_text}",
            f"库存状态: {stock_text}",
            f"尺寸规格: {size_text}",
            f"净重: {weight_text}",
            f"产品描述: {description}",
        ]
        if product.get("video_url"):
            lines.append(f"介绍视频: {product['video_url']}")
        lines.append("="*60)
        return "\n".join(lines)

    def select_product(self, category_name: str, user_input: str) -> Optional[dict]:
        products = self.list_products(category_name)
        if not products:
            return None
        raw = user_input.strip()
        if raw.isdigit():
            index = int(raw)
            if 1 <= index <= len(products):
                return products[index - 1]
            return None
        normalized_target = self._normalize_text(raw)
        best_match = None
        best_score = 0.0
        for product in products:
            normalized_name = self._normalize_text(product["name"])
            if normalized_target in normalized_name or normalized_name in normalized_target:
                return product
            score = SequenceMatcher(None, normalized_target, normalized_name).ratio()
            if score > best_score:
                best_score = score
                best_match = product
        return best_match if best_score >= 0.55 else None

# 初始化引擎（如果数据文件存在）
PRODUCT_JSON_PATH = "03_PRODUCT_INFO.json"
CATEGORY_CSV_PATH = "01_PRODUCT_MASTER_BASE.csv"
product_browse_engine = None
if Path(PRODUCT_JSON_PATH).exists() and Path(CATEGORY_CSV_PATH).exists():
    product_browse_engine = ProductBrowseEngine(
        product_json_path=PRODUCT_JSON_PATH,
        category_csv_path=CATEGORY_CSV_PATH,
    )
else:
    print(f"警告: 产品数据文件不存在，产品浏览功能将不可用")


ProductBrowseEngine 初始化完成
产品分类数: 19
总产品数: 516


In [ ]:
#产品浏览会话
def run_product_browse_session() -> dict:
    browse_state = {
        "selected_category": "",
        "viewed_products": [],
        "is_complete": False,
    }
    if not product_browse_engine:
        print("\n产品浏览引擎未初始化，无法浏览产品。")
        return browse_state
    print("\n" + "="*55)
    print("产品浏览模式")
    print("输入分类名称或编号选择分类，0 返回，q 退出")
    print("="*55)
    categories = product_browse_engine.get_categories()
    print("\n可选分类：")
    for idx, category in enumerate(categories, 1):
        print(f"{idx}. {category}")
    while not browse_state["selected_category"]:
        raw = input("\n请选择分类: ").strip()
        if not raw:
            print("输入不能为空")
            continue
        if raw.lower() in {"q", "quit", "exit"}:
            browse_state["is_complete"] = True
            return browse_state
        category, method = product_browse_engine.match_category(raw)
        if not category:
            print("未能匹配到分类，请重新输入")
            continue
        if method in ("llm", "fallback"):
            print(f"已匹配到分类: {category}")
        browse_state["selected_category"] = category
        print(product_browse_engine.format_product_list(category))
    while not browse_state["is_complete"]:
        raw = input("\n请输入产品编号或名称 (0 返回分类, q 退出): ").strip()
        if not raw:
            print("输入不能为空")
            continue
        if raw.lower() in {"q", "quit", "exit"}:
            browse_state["is_complete"] = True
            break
        if raw == "0":
            browse_state["selected_category"] = ""
            print("\n可选分类：")
            for idx, category in enumerate(categories, 1):
                print(f"{idx}. {category}")
            while not browse_state["selected_category"]:
                category_input = input("\n请选择分类: ").strip()
                if not category_input:
                    print("输入不能为空")
                    continue
                if category_input.lower() in {"q", "quit", "exit"}:
                    browse_state["is_complete"] = True
                    return browse_state
                category, method = product_browse_engine.match_category(category_input)
                if not category:
                    print("未能匹配到分类，请重新输入")
                    continue
                browse_state["selected_category"] = category
                print(product_browse_engine.format_product_list(category))
            continue
        product = product_browse_engine.select_product(browse_state["selected_category"], raw)
        if not product:
            print("未能找到该产品，请重新输入")
            continue
        browse_state["viewed_products"].append(product["name"])
        print(product_browse_engine.format_product_detail(product))
        print("\n输入 0 返回分类，q 退出，或其他编号继续查看")
    return browse_state


In [ ]:
#护理咨询会话 (RAG)
def run_nursing_consultation_session() -> dict:
    consultation_state = {
        "qa_history": [],
        "is_complete": False
    }
    print("\n" + "="*55)
    print("护理咨询模式")
    print("输入您的问题，0 返回主菜单，q 退出")
    print("="*55)
    try:
        nursing_vectorstore = Chroma(
            persist_directory="./nursing_chroma_db",
            collection_name="nursing_consultation",
            embedding_function=embeddings
        )
        count = nursing_vectorstore._collection.count()
        print(f"护理知识库加载完成，共 {count} 条文档\n")
    except Exception as e:
        print(f"护理向量库加载失败: {e}")
        print("请确保护理文档已提前构建向量索引到 ./nursing_chroma_db\n")
        return consultation_state
    while not consultation_state["is_complete"]:
        user_question = input("\n请输入护理问题 (0 返回, q 退出): ").strip()
        if not user_question:
            continue
        if user_question.lower() in {"q", "quit", "exit"}:
            consultation_state["is_complete"] = True
            break
        if user_question == "0":
            print("\n返回主菜单")
            break
        print("正在检索相关知识...")
        try:
            docs = nursing_vectorstore.similarity_search(user_question, k=5)
            if not docs:
                print("\n未找到相关护理资料，建议咨询专业医护人员。")
                continue
            context_text = "\n\n".join([f"参考 {i+1}: {doc.page_content}" for i, doc in enumerate(docs)])
            user_prompt = NURSING_CONSULT_USER_TEMPLATE.format(
                context=context_text,
                user_input=user_question
            )
            messages = [
                SystemMessage(content=NURSING_CONSULT_SYSTEM_PROMPT),
                HumanMessage(content=user_prompt)
            ]
            response = llm.invoke(messages).content.strip()
            print(f"\n护理顾问回复:\n{response}")
            print("\n" + "-"*55)
            consultation_state["qa_history"].append({
                "question": user_question,
                "answer": response
            })
        except Exception as e:
            print(f"\n咨询过程出错: {e}")
    return consultation_state


In [ ]:
#初始引导 & 核心意图路由 (handle_choice_node_v2)
def initial_guide_node(state: AgentState) -> AgentState:
    guide_text = """
請問有什麼能幫您的？
請直接輸入數字選擇您的需求：
1. 護理諮詢：護理方法、輔具使用、照顧技巧、長者健康
2. 產品-問題解決型：有具體痛點，需要個人化輔具推薦
3. 產品-瀏覽了解型：只想了解產品、查看型號種類
4. 其他問題：自主描述您的需求（自由輸入）
"""
    state["messages"].append({"role": "assistant", "content": guide_text})
    state["is_intent_clear"] = False
    return state

def handle_choice_node_v2(state: AgentState) -> AgentState:
    def _launch_logic_rule(state: AgentState) -> AgentState:
        lr_final = run_logic_rule_session()
        state["logic_rule_qa_history"] = lr_final["qa_history"]
        state["collected_recommendations"] = lr_final["collected_recommendations"]
        recs = logic_engine.get_recommendation_summary(lr_final["collected_recommendations"]) if logic_engine else []
        rec_lines = "\n".join([f"• {r['content']} (设备标签: {r['device_tag']})" for r in recs])
        state["messages"].append({
            "role": "assistant",
            "content": f"设备租赁评估完成，推荐如下:\n{rec_lines}"
        })
        return state

    def _launch_product_browse(state: AgentState) -> AgentState:
        browse_final = run_product_browse_session()
        state["browse_selected_category"] = browse_final["selected_category"]
        state["browse_viewed_products"] = browse_final["viewed_products"]
        viewed_lines = "\n".join([f"• {name}" for name in browse_final["viewed_products"]])
        if not viewed_lines:
            viewed_lines = "• 未查看具体产品"
        state["messages"].append({
            "role": "assistant",
            "content": (
                f"产品浏览会话结束\n"
                f"选中分类: {browse_final['selected_category'] or '无'}\n"
                f"查看过的产品:\n{viewed_lines}"
            )
        })
        return state

    def _launch_nursing_consultation(state: AgentState) -> AgentState:
        nursing_final = run_nursing_consultation_session()
        state["nursing_qa_history"] = nursing_final["qa_history"]
        qa_count = len(nursing_final["qa_history"])
        state["messages"].append({
            "role": "assistant",
            "content": f"护理咨询会话结束，共进行 {qa_count} 轮问答。"
        })
        return state

    user_choice = state["user_input"].strip()
    if user_choice == "1":
        state["intent"] = "护理咨询"
        state["is_intent_clear"] = True
        state["messages"].append({"role": "assistant", "content": "正在启动护理咨询..."})
        state = _launch_nursing_consultation(state)
    elif user_choice == "2":
        state["intent"] = "设备租赁"
        state["is_intent_clear"] = True
        state["messages"].append({"role": "assistant", "content": "正在启动设备租赁评估..."})
        state = _launch_logic_rule(state)
    elif user_choice == "3":
        state["intent"] = "产品浏览"
        state["is_intent_clear"] = True
        state["messages"].append({"role": "assistant", "content": "正在启动产品浏览..."})
        state = _launch_product_browse(state)
    elif user_choice == "4":
        print("\n请直接输入您的需求描述：")
        free_input = input("请直接输入您的需求描述: ").strip()
        prompt = INTENT_CLASSIFY_PROMPT.format(user_input=free_input)
        intent_result = llm.invoke(prompt).content.strip()
        if intent_result == "产品-问题解决型":
            state["intent"] = intent_result
            state["is_intent_clear"] = True
            state["messages"].append({"role": "assistant", "content": f"识别意图: {intent_result}，启动设备评估..."})
            state = _launch_logic_rule(state)
        elif intent_result == "产品-浏览了解型":
            state["intent"] = intent_result
            state["is_intent_clear"] = True
            state["messages"].append({"role": "assistant", "content": f"识别意图: {intent_result}，启动产品浏览..."})
            state = _launch_product_browse(state)
        elif intent_result == "护理咨询":
            state["intent"] = intent_result
            state["is_intent_clear"] = True
            state["messages"].append({"role": "assistant", "content": f"识别意图: {intent_result}，启动护理咨询..."})
            state = _launch_nursing_consultation(state)
        else:
            state["is_intent_clear"] = False
            state["messages"].append({
                "role": "assistant",
                "content": "未能识别您的意图，请直接输入 1/2/3/4 选择服务。"
            })
    else:
        state["is_intent_clear"] = False
        state["messages"].append({
            "role": "assistant",
            "content": "输入无效，请输入 1/2/3/4 选择对应服务。"
        })
    return state


In [ ]:
# CATEGORY_CONFIG & 设备跟进函数
PRODUCT_REFINE_LOGIC_PATH = "04_PRODUCT_REFINE_LOGIC.json"

def load_product_refine_logic(json_path: str = PRODUCT_REFINE_LOGIC_PATH) -> Dict[str, dict]:
    """加载产品精细筛选配置。"""
    if not Path(json_path).exists():
        raise FileNotFoundError(f"找不到产品精细筛选配置文件: {json_path}")
    with open(json_path, "r", encoding="utf-8") as f:
        config = json.load(f)
    if not isinstance(config, dict):
        raise ValueError("产品精细筛选配置必须是 JSON object")
    return config

CATEGORY_CONFIG = load_product_refine_logic()
print(f"产品精细筛选配置加载完成: {len(CATEGORY_CONFIG)} 个分类")

def _print_banner(title, tag):
    print(f"\n{'='*60}")
    print(f"【{title}】 标签: {tag}")
    print(f"{'='*60}")

def _parse_choice(user_input, options):
    user_input = user_input.strip()
    if not user_input:
        return None, False, ""
    selected = next((o for o in options if o["label"] == user_input), None)
    if selected:
        return selected, False, ""
    selected = next((o for o in options if user_input in o["text"] or o["text"] in user_input), None)
    if selected:
        return selected, False, ""
    return None, True, user_input

def _prompt_custom_need(prompt_text="请描述您的具体需求："):
    while True:
        custom_text = input(prompt_text).strip()
        if custom_text:
            return custom_text
        print("输入不能为空，请重新输入")

def _trigger_inventory_search(tag, category_name, user_desc=""):
    print("\n触发库存检索...")
    return {
        "trigger_inventory_search": True,
        "tag": tag,
        "category": category_name,
        "user_choice": user_desc
    }

def _ask_and_recommend(options, question_text, category_name, tag):
    print(f"\n【{question_text}】")
    for opt in options:
        print(f"{opt['label']}. {opt['text']}")
    raw_input = input("请输入选项: ").strip()
    selected, is_custom, custom_text = _parse_choice(raw_input, options)
    if is_custom:
        return _trigger_inventory_search(tag=tag, category_name=category_name, user_desc=f"自由输入: {custom_text}")
    if not selected:
        print("未能识别选项")
        return None
    if selected.get("is_other"):
        custom_text = _prompt_custom_need("选择了'其他'，请补充说明：")
        return _trigger_inventory_search(tag=tag, category_name=category_name, user_desc=f"其他: {custom_text}")
    if "followup" in selected:
        fu = selected["followup"]
        print(f"\n{fu['question']}")
        for sub in fu["options"]:
            print(f"{sub['label']}. {sub['text']}")
        sub_raw = input("请选择: ").strip()
        sub, sub_is_custom, sub_custom_text = _parse_choice(sub_raw, fu["options"])
        if sub_is_custom:
            return _trigger_inventory_search(tag=tag, category_name=category_name, user_desc=f"{selected['text']}-{sub_custom_text}")
        if not sub:
            print("选项识别失败")
            return None
        if sub.get("is_other"):
            custom_text = _prompt_custom_need("请补充说明：")
            return _trigger_inventory_search(tag=tag, category_name=category_name, user_desc=f"{selected['text']}-{custom_text}")
        recommend = sub["recommend"]
        print(f"\n推荐产品: {recommend}")
        return {"tag": tag, "category": category_name, "choice": selected["text"], "sub_choice": sub["text"], "recommend": recommend}
    recommend = selected["recommend"]
    print(f"\n推荐产品: {recommend}")
    return {"tag": tag, "category": category_name, "choice": selected["text"], "recommend": recommend}

def _handle_no_product(cfg, tag):
    print(f"\n【{cfg['question']}】")
    options = cfg.get("options", [])
    for opt in options:
        print(f"{opt['label']}. {opt['text']}")
    if options:
        raw_input = input("请输入选项: ").strip()
        selected, is_custom, custom_text = _parse_choice(raw_input, options)
        if is_custom or (selected and selected.get("is_other")):
            custom_text = _prompt_custom_need("请描述需求：") if selected and selected.get("is_other") else custom_text
            return _trigger_inventory_search(tag=tag, category_name=cfg["name"], user_desc=f"{custom_text}")
    print(f"\n{'='*50}")
    print("服务指引")
    print(f"【{cfg['name']}】暂无直接产品推荐")
    print("="*50)
    return {"tag": tag, "category": cfg["name"], "mode": "no_product"}

def _handle_redirect(cfg, tag):
    print(f"\n【{cfg['question']}】")
    options = cfg.get("options", [])
    for opt in options:
        print(f"{opt['label']}. {opt['text']}")
    if options:
        raw_input = input("请输入选项: ").strip()
        selected, is_custom, custom_text = _parse_choice(raw_input, options)
        if is_custom:
            return _trigger_inventory_search(tag=tag, category_name=cfg["name"], user_desc=f"{custom_text}")
        if selected and selected.get("is_other"):
            custom_text = _prompt_custom_need("请补充说明：")
            return _trigger_inventory_search(tag=tag, category_name=cfg["name"], user_desc=f"{custom_text}")
    print(f"\n{'='*50}")
    print("转介服务")
    print(f"【{cfg['name']}】需要人工或第三方服务")
    print("="*50)
    return {"tag": tag, "category": cfg["name"], "mode": "redirect"}

def run_device_followup(device_tags):
    results = []
    for tag in device_tags:
        cfg = CATEGORY_CONFIG.get(tag)
        if not cfg:
            print(f"\n未找到标签 {tag} 的配置")
            continue
        _print_banner(cfg["name"], tag)
        if cfg["mode"] == "no_product":
            res = _handle_no_product(cfg, tag)
            results.append(res)
        elif cfg["mode"] == "redirect":
            res = _handle_redirect(cfg, tag)
            results.append(res)
        elif cfg["mode"] in ("recommend", "nested"):
            res = _ask_and_recommend(
                options=cfg["options"],
                question_text=cfg["question"],
                category_name=cfg["name"],
                tag=tag
            )
            if res:
                results.append(res)
        else:
            print(f"未知模式: {cfg['mode']}")
    return results


In [ ]:
#库存搜索函数
PRODUCT_CSV_PATH = Path("01_PRODUCT_MASTER_BASE.csv")
PRODUCT_CATALOG = None

def load_product_catalog(csv_path: Path = PRODUCT_CSV_PATH):
    global PRODUCT_CATALOG
    if PRODUCT_CATALOG is not None:
        return PRODUCT_CATALOG
    if not csv_path.exists():
        print(f"警告: 找不到产品目录 {csv_path}")
        PRODUCT_CATALOG = []
        return PRODUCT_CATALOG
    with csv_path.open("r", encoding="utf-8-sig", newline="") as f:
        reader = csv.DictReader(f)
        PRODUCT_CATALOG = list(reader)
    return PRODUCT_CATALOG

def _safe_text(value):
    text = str(value).strip() if value is not None else ""
    return text if text else "暂无"

def _format_dimensions(row: dict) -> str:
    length = str(row.get("dimension_length", "")).strip()
    width = str(row.get("dimension_width", "")).strip()
    height = str(row.get("dimension_height", "")).strip()
    parts = [v for v in [length, width, height] if v]
    if not parts:
        return "暂无尺寸"
    if len(parts) == 3:
        return f"{length} x {width} x {height}"
    return " x ".join(parts)

def _extract_recommend_tokens(recommend_text: str) -> list:
    if not recommend_text:
        return []
    parts = re.split(r"[\/|]|、|,|;|；", str(recommend_text))
    tokens = []
    seen = set()
    for part in parts:
        token = part.strip()
        if len(token) < 2:
            continue
        if token not in seen:
            seen.add(token)
            tokens.append(token)
    return tokens

def search_inventory_strict(tag: str, category: str = "", recommend_text: str = "", top_k: int = 5):
    catalog = load_product_catalog()
    if not catalog:
        return []
    category_id = str(tag).strip()
    filtered = [row for row in catalog if str(row.get("category_id", "")).strip() == category_id]
    if not filtered and category:
        filtered = [row for row in catalog if str(row.get("category_name", "")).strip() == category.strip()]
    tokens = _extract_recommend_tokens(recommend_text)
    if not tokens:
        return []
    matches = []
    for row in filtered:
        product_name = str(row.get("product_name", "")).strip().lower()
        description = str(row.get("description", "")).strip().lower()
        for token in tokens:
            t = token.lower()
            if t in product_name or t in description:
                matches.append(row)
                break
    return matches[:top_k]

def _summarize_row_for_llm(row: dict, index: int) -> str:
    product_name = _safe_text(row.get("product_name"))
    stock_status = _safe_text(row.get("stock_status"))
    dimensions = _format_dimensions(row)
    net_weight = _safe_text(row.get("net_weight"))
    description = _safe_text(row.get("description"))
    if len(description) > 220:
        description = description[:220] + "..."
    return (
        f"[{index}] 产品: {product_name}\n"
        f"库存: {stock_status}\n"
        f"尺寸: {dimensions}\n"
        f"重量: {net_weight}\n"
        f"描述: {description}"
    )

def _extract_indices_from_text(raw: str, max_index: int) -> list:
    if not raw:
        return []
    text = raw.strip()
    text = re.sub(r"^```(?:json)?\s*", "", text, flags=re.I)
    text = re.sub(r"\s*```$", "", text)
    match = re.search(r"\{.*\}", text, re.S)
    if match:
        try:
            data = json.loads(match.group(0))
            indices = data.get("indices", [])
            cleaned = []
            seen = set()
            for idx in indices:
                if isinstance(idx, int) and 1 <= idx <= max_index and idx not in seen:
                    seen.add(idx)
                    cleaned.append(idx)
            if cleaned:
                return cleaned
        except Exception:
            pass
    nums = re.findall(r"\d+", text)
    cleaned = []
    seen = set()
    for n in nums:
        idx = int(n)
        if 1 <= idx <= max_index and idx not in seen:
            seen.add(idx)
            cleaned.append(idx)
    return cleaned

def _fallback_keyword_products(candidates: list, user_desc: str, top_k: int = 3) -> list:
    keywords = _extract_recommend_tokens(user_desc)
    if not keywords:
        return candidates[:top_k]
    ranked = []
    for row in candidates:
        text = (str(row.get("product_name", "")) + " " + str(row.get("description", ""))).lower()
        score = sum(1 for kw in keywords if kw.lower() in text)
        ranked.append((score, row))
    ranked.sort(key=lambda x: x[0], reverse=True)
    picked = [row for score, row in ranked if score > 0]
    return picked[:top_k] if picked else candidates[:top_k]

def search_inventory_with_llm(tag: str, category: str = "", user_desc: str = "", top_k: int = 3):
    catalog = load_product_catalog()
    if not catalog:
        return []
    category_id = str(tag).strip()
    candidates = [row for row in catalog if str(row.get("category_id", "")).strip() == category_id]
    if not candidates and category:
        candidates = [row for row in catalog if str(row.get("category_name", "")).strip() == category.strip()]
    if not candidates:
        return []
    indexed_candidates = candidates[:20]
    candidate_blocks = [_summarize_row_for_llm(row, i) for i, row in enumerate(indexed_candidates, 1)]
    prompt = f"""
你是库存匹配助手。请根据用户需求，从以下候选产品中选出最匹配的 {top_k} 款。
要求：
1. 只考虑候选列表中的产品
2. 返回最匹配的 {top_k} 个产品编号
3. 如果没有匹配的，返回空列表
4. 严格按 JSON 格式输出: {{"indices":[1,2]}}
5. 也可以直接回复编号如 1,3
6. 不要输出解释

产品分类: {category}
用户需求: {user_desc}

候选产品:
{chr(10).join(candidate_blocks)}
""".strip()
    try:
        raw = llm.invoke(prompt).content.strip()
        print("LLM 库存匹配结果:", repr(raw))
        indices = _extract_indices_from_text(raw, max_index=len(indexed_candidates))
        if indices:
            picked = []
            seen = set()
            for idx in indices:
                if idx not in seen:
                    seen.add(idx)
                    picked.append(indexed_candidates[idx - 1])
            return picked[:top_k]
        print("LLM 未返回有效索引，回退到关键词匹配")
    except Exception as e:
        print(f"LLM 库存匹配失败: {e}")
    return _fallback_keyword_products(indexed_candidates, user_desc=user_desc, top_k=top_k)

def _format_sales_price(value) -> str:
    if value in (None, ""):
        return "价格待定"
    try:
        return f"HK${float(value):.0f}"
    except (TypeError, ValueError):
        return _safe_text(value)


INVENTORY_PRODUCT_DETAILS = None


def _normalize_product_name(text: str) -> str:
    text = (text or "").strip().lower()
    replacements = {"（": "(", "）": ")", "-": "", "_": "", "，": "", ",": "", " ": ""}
    for old, new in replacements.items():
        text = text.replace(old, new)
    return "".join(text.split())


def _strip_product_html(text: str) -> str:
    if not text:
        return ""
    cleaned = re.sub(r"<br\s*/?>", "\n", text, flags=re.IGNORECASE)
    cleaned = re.sub(r"</p>|</li>|</h\d>", "\n", cleaned, flags=re.IGNORECASE)
    cleaned = re.sub(r"<[^>]+>", "", cleaned)
    cleaned = html.unescape(cleaned)
    cleaned = re.sub(r"\s+", "", cleaned)
    return cleaned.strip()


def _load_inventory_product_details() -> list:
    global INVENTORY_PRODUCT_DETAILS
    if INVENTORY_PRODUCT_DETAILS is not None:
        return INVENTORY_PRODUCT_DETAILS
    detail_path = Path("03_PRODUCT_INFO.json")
    if not detail_path.exists():
        INVENTORY_PRODUCT_DETAILS = []
        return INVENTORY_PRODUCT_DETAILS
    with detail_path.open("r", encoding="utf-8") as f:
        INVENTORY_PRODUCT_DETAILS = json.load(f)
    return INVENTORY_PRODUCT_DETAILS


def _best_inventory_detail_match(product_name: str) -> dict:
    normalized_target = _normalize_product_name(product_name)
    best_match = None
    best_score = 0.0
    for item in _load_inventory_product_details():
        candidate_name = item.get("Name", "")
        normalized_candidate = _normalize_product_name(candidate_name)
        if not normalized_candidate:
            continue
        if normalized_target == normalized_candidate:
            return item
        if normalized_target in normalized_candidate or normalized_candidate in normalized_target:
            score = 0.95
        else:
            score = SequenceMatcher(None, normalized_target, normalized_candidate).ratio()
        if score > best_score:
            best_score = score
            best_match = item
    return best_match if best_score >= 0.62 else {}


def _get_inventory_product_detail(row: dict) -> dict:
    """补齐推荐租赁产品的价格、视频和更完整描述。"""
    product_name = row.get("product_name", "")
    detail = _best_inventory_detail_match(product_name)

    description = ""
    if detail:
        description = _strip_product_html(detail.get("eCommerce Description", ""))
        description = description or (detail.get("Product/Description") or "").strip()
    description = description or (row.get("description") or "").strip()
    if len(description) > 360:
        description = description[:360].rstrip() + "..."

    dimension_parts = []
    length = detail.get("Dimension Length") or row.get("dimension_length")
    width = detail.get("Dimension Width") or row.get("dimension_width")
    height = detail.get("Dimension Height") or row.get("dimension_height")
    if length:
        dimension_parts.append(f"长{length}")
    if width:
        dimension_parts.append(f"宽{width}")
    if height:
        dimension_parts.append(f"高{height}")
    dimension_text = "/".join(dimension_parts) if dimension_parts else _format_dimensions(row)

    return {
        "product_name": _safe_text(product_name),
        "category_name": _safe_text(row.get("category_name")),
        "sales_price": _format_sales_price(detail.get("Sales Price") or row.get("sales_price")),
        "stock_status": _safe_text(row.get("stock_status")),
        "dimension": dimension_text,
        "net_weight": _safe_text(detail.get("Net Weight") or row.get("net_weight")),
        "description": description or "暂无描述",
        "video_url": _safe_text(detail.get("Introduction Video URL")),
    }
def display_inventory_results(tag: str, category: str, results: list,
                              recommend_text: str = "", user_desc: str = "",
                              trigger_inventory_search: bool = False):
    if not results and not trigger_inventory_search:
        return
    print("\n" + "="*60)
    print(f"【库存检索结果】分类: {category} | 标签: {tag}")
    if recommend_text:
        print(f"推荐关键词: {recommend_text}")
    if user_desc:
        print(f"用户描述: {user_desc}")
    print("="*60)
    if not results:
        print("未找到匹配产品")
        return
    for i, row in enumerate(results, 1):
        detail = _get_inventory_product_detail(row)
        print(f"{i}. 产品名称: {detail['product_name']}")
        print(f"   所属分类: {detail['category_name']}")
        print(f"   销售价格: {detail['sales_price']}")
        print(f"   库存状态: {detail['stock_status']}")
        print(f"   尺寸规格: {detail['dimension']}")
        print(f"   净重: {detail['net_weight']}")
        print(f"   产品描述: {detail['description']}")
        print(f"   介绍视频: {detail['video_url']}")
        print("-"*60)
def build_inventory_tasks(followup_results):
    tasks = []
    for item in followup_results:
        if not item:
            continue
        user_desc = item.get("user_choice", "")
        if not user_desc:
            pieces = [item.get("choice", ""), item.get("sub_choice", "")]
            user_desc = "/".join([p for p in pieces if p])
        tasks.append({
            "tag": item.get("tag", ""),
            "category": item.get("category", ""),
            "recommend_text": item.get("recommend", ""),
            "user_desc": user_desc,
            "trigger_inventory_search": item.get("trigger_inventory_search", False),
        })
    return tasks


In [ ]:
#主程序入口
if __name__ == "__main__":
    print("="*60)
    print("智能助手 v3.0 | 护理咨询 + 设备租赁 + 产品浏览")
    print("="*60)

    required_names = [
        "initial_guide_node", "handle_choice_node_v2", "run_device_followup",
        "logic_engine", "llm", "run_logic_rule_session",
        "run_product_browse_session", "run_nursing_consultation_session"
    ]
    missing = [name for name in required_names if name not in globals()]
    if missing:
        raise RuntimeError(f"缺少必要函数/对象: {', '.join(missing)}")

    def create_session_state() -> dict:
        """每次回到主菜单时创建一个新的分支状态，避免重复处理上一轮结果。"""
        return {
            "messages": [],
            "user_input": "",
            "intent": "",
            "is_intent_clear": False,
            "logic_rule_qa_history": [],
            "collected_recommendations": [],
            "browse_selected_category": "",
            "browse_viewed_products": [],
            "nursing_qa_history": [],
            "followup_results": [],
        }

    def run_post_tasks(state: dict) -> dict:
        """设备租赁分支完成后，继续执行设备跟进和库存检索。"""
        if not state.get("collected_recommendations"):
            return state

        print("\n" + "="*60)
        print("正在处理后续任务...")
        print("="*60)

        recs = logic_engine.get_recommendation_summary(state["collected_recommendations"]) if logic_engine else []
        device_tags = [r["device_tag"] for r in recs if r.get("device_tag")]
        if not device_tags:
            return state

        followup_results = run_device_followup(device_tags)
        state["followup_results"] = followup_results

        inventory_search_tasks = build_inventory_tasks(followup_results)
        if inventory_search_tasks:
            print("\n" + "="*60)
            print("开始库存检索...")
            print("="*60)

            for task in inventory_search_tasks:
                print(f"\n→ 检索分类: {task['tag']} | {task['category']} ...")

                if task.get("trigger_inventory_search"):
                    matches = search_inventory_with_llm(
                        tag=task["tag"],
                        category=task["category"],
                        user_desc=task.get("user_desc", "")
                    )
                else:
                    matches = search_inventory_strict(
                        tag=task["tag"],
                        category=task["category"],
                        recommend_text=task.get("recommend_text", "")
                    )

                display_inventory_results(
                    tag=task["tag"],
                    category=task["category"],
                    results=matches,
                    recommend_text=task.get("recommend_text", ""),
                    user_desc=task.get("user_desc", ""),
                    trigger_inventory_search=task.get("trigger_inventory_search", False)
                )

        return state

    def print_session_summary(state: dict):
        print("\n" + "="*60)
        print("本次分支总结")
        print("="*60)

        if state.get("nursing_qa_history"):
            n = len(state["nursing_qa_history"])
            print(f"\n【护理咨询】已完成 {n} 轮问答")
            print("历史记录: state['nursing_qa_history']")

        if state.get("browse_selected_category") or state.get("browse_viewed_products"):
            print(f"\n【产品浏览】分类: {state.get('browse_selected_category', '无')}")
            print(f"查看产品数: {len(state.get('browse_viewed_products', []))}")
            print("查看列表: state['browse_viewed_products']")

        if state.get("logic_rule_qa_history"):
            n = len(state["logic_rule_qa_history"])
            print(f"\n【设备租赁评估】已完成 {n} 轮问答")
            print("问答历史: state['logic_rule_qa_history']")
            print("推荐节点: state['collected_recommendations']")

        if state.get("followup_results"):
            print(f"\n【设备跟进】已处理 {len(state['followup_results'])} 个标签")

    def ask_need_more_help() -> bool:
        """分支结束后询问是否回到主菜单。True 表示继续，False 表示结束。"""
        yes_inputs = {"有", "是", "需要", "要", "继续", "yes", "y", "1"}
        no_inputs = {"没有", "无", "不用", "不需要", "否", "不了", "no", "n", "0", "结束"}

        while True:
            raw = input("\n请问还有其他需要帮助的吗？(有/没有): ").strip().lower()

            if raw in yes_inputs:
                return True

            if raw in no_inputs:
                return False

            print("请输入“有”或“没有”。")

    session_records = []

    while True:
        state = create_session_state()

        state = initial_guide_node(state)
        print("\n助手:", state["messages"][-1]["content"])

        while True:
            user_input = input("\n请输入 (1/2/3/4): ").strip()

            if not user_input:
                print("输入不能为空")
                continue

            state["user_input"] = user_input
            state = handle_choice_node_v2(state)
            last_msg = state["messages"][-1]["content"]

            interactive_intents = {"设备租赁", "产品浏览", "护理咨询"}

            if not (state["is_intent_clear"] and state["intent"] in interactive_intents):
                print("\n助手:", last_msg)

            if state["is_intent_clear"]:
                break

        state = run_post_tasks(state)
        print_session_summary(state)
        session_records.append(state)

        if ask_need_more_help():
            print("\n好的，我们回到主菜单。")
            continue

        break

    print("\n" + "="*60)
    print(f"会话结束，共完成 {len(session_records)} 个分支。")
    print("感谢使用，祝您健康！")
    print("="*60)


智能助手 v3.0 | 护理咨询 + 设备租赁 + 产品浏览

助手: 
請問有什麼能幫您的？
請直接輸入數字選擇您的需求：
1. 護理諮詢：護理方法、輔具使用、照顧技巧、長者健康
2. 產品-問題解決型：有具體痛點，需要個人化輔具推薦
3. 產品-瀏覽了解型：只想了解產品、查看型號種類
4. 其他問題：自主描述您的需求（自由輸入）


护理咨询模式
输入您的问题，0 返回主菜单，q 退出
护理知识库加载完成，共 364 条文档

正在检索相关知识...

护理顾问回复:
抱歉，目前的护理知识库中暂未收录关于此问题的资料。建议您咨询专业医生或复康护士。

温馨提示：以上建议仅供长者护理参考，不替代专业医疗意见，如有健康相关疑问，请及时咨询医护人员。

-------------------------------------------------------

本次分支总结

【护理咨询】已完成 1 轮问答
历史记录: state['nursing_qa_history']

会话结束，共完成 1 个分支。
感谢使用，祝您健康！
